# LeetCode 223 — Rectangle Area bằng Python

Link đề: https://leetcode.com/problems/rectangle-area/ · Độ khó: **Medium** · Chủ đề: Math, Geometry (hình học tọa độ)

Skeleton LeetCode:

```python
class Solution:
    def computeArea(self, ax1: int, ay1: int, ax2: int, ay2: int,
                    bx1: int, by1: int, bx2: int, by2: int) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ bản mô tả gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force độc lập (đếm từng ô đơn vị trong hộp bao).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho hai hình chữ nhật có cạnh song song hai trục tọa độ (**rectilinear**), tính **tổng diện tích mà hai hình phủ lên mặt phẳng** — phần giao nhau chỉ được tính **một lần**.
2. Nắm nguyên lý **bao hàm – loại trừ (inclusion–exclusion)**: `|A ∪ B| = |A| + |B| - |A ∩ B|`.
3. Biết tính **phần giao** của hai hình chữ nhật bằng cách tính **độ chồng lấn trên từng trục**: `max(0, min(x2) - max(x1))` rồi nhân hai trục với nhau.
4. Đi từ cách **đếm ô đơn vị** → **bao hàm – loại trừ có kẹp `max(0, …)`** → **tách trường hợp rõ ràng** → **hàm chồng lấn 1 chiều tái sử dụng** → **nén tọa độ** (dễ mở rộng ra `n` hình chữ nhật).
5. Tránh các lỗi kinh điển: **quên kẹp `max(0, …)`** (hai số âm nhân ra số dương!), nhầm diện tích hình với tọa độ, giả định hình `A` luôn nằm bên trái/dưới `B`, và quên trường hợp **chỉ chạm cạnh**.

## 2. Đề bài & ví dụ

Cho tọa độ của hai hình chữ nhật có các cạnh song song với trục tọa độ trong mặt phẳng 2 chiều. Trả về **tổng diện tích** do hai hình chữ nhật phủ lên.

- Hình thứ nhất xác định bởi góc **dưới-trái** `(ax1, ay1)` và góc **trên-phải** `(ax2, ay2)`.
- Hình thứ hai xác định bởi góc **dưới-trái** `(bx1, by1)` và góc **trên-phải** `(bx2, by2)`.

**Ví dụ 1**

- Input: `ax1 = -3, ay1 = 0, ax2 = 3, ay2 = 4, bx1 = 0, by1 = -1, bx2 = 9, by2 = 2`
- Output: `45`
- Giải thích: hình A rộng `6`, cao `4` (diện tích `24`); hình B rộng `9`, cao `3` (diện tích `27`); phần giao rộng `3`, cao `2` (diện tích `6`); `24 + 27 - 6 = 45`.

**Ví dụ 2**

- Input: `ax1 = -2, ay1 = -2, ax2 = 2, ay2 = 2, bx1 = -2, by1 = -2, bx2 = 2, by2 = 2`
- Output: `16`
- Giải thích: hai hình trùng nhau hoàn toàn, tổng diện tích phủ là `4 × 4 = 16`.

**Ràng buộc:** `-10^4 <= ax1 <= ax2 <= 10^4`; `-10^4 <= ay1 <= ay2 <= 10^4`; `-10^4 <= bx1 <= bx2 <= 10^4`; `-10^4 <= by1 <= by2 <= 10^4`.

(Lưu ý: ràng buộc dùng `<=` nên hình chữ nhật có thể **suy biến** thành đoạn thẳng hoặc điểm — diện tích `0`.)

In [1]:
from typing import List
import random

examples = [
    ((-3, 0, 3, 4, 0, -1, 9, 2), 45),
    ((-2, -2, 2, 2, -2, -2, 2, 2), 16),
]
for args, expected in examples:
    print(f"(ax1,ay1,ax2,ay2, bx1,by1,bx2,by2)={args} -> kỳ vọng {expected}")

(ax1,ay1,ax2,ay2, bx1,by1,bx2,by2)=(-3, 0, 3, 4, 0, -1, 9, 2) -> kỳ vọng 45
(ax1,ay1,ax2,ay2, bx1,by1,bx2,by2)=(-2, -2, 2, 2, -2, -2, 2, 2) -> kỳ vọng 16


## 3. Kiến thức nền

### 3.1. Hình chữ nhật cạnh song song trục: chiều rộng, chiều cao, diện tích

Hình chữ nhật có góc dưới-trái `(x1, y1)` và trên-phải `(x2, y2)` (với `x1 <= x2`, `y1 <= y2`) có chiều rộng `x2 - x1`, chiều cao `y2 - y1`, diện tích `(x2 - x1) * (y2 - y1)`. Chú ý **dùng hiệu tọa độ**, không dùng chính tọa độ (tọa độ có thể âm).

In [2]:
ax1, ay1, ax2, ay2 = -3, 0, 3, 4
print("rộng:", ax2 - ax1, "cao:", ay2 - ay1, "diện tích:", (ax2 - ax1) * (ay2 - ay1))   # 6 4 24
bx1, by1, bx2, by2 = 0, -1, 9, 2
print("diện tích B:", (bx2 - bx1) * (by2 - by1))                                         # 27

rộng: 6 cao: 4 diện tích: 24
diện tích B: 27


### 3.2. Bao hàm – loại trừ (inclusion–exclusion)

Khi cộng diện tích hai hình, vùng giao nhau bị **đếm hai lần**. Muốn đếm một lần thì **trừ đi một lần**:

`diện tích phủ = |A| + |B| - |A ∩ B|`.

Hình dung như hai tờ giấy đè lên nhau: tổng diện tích mặt bàn bị che = hai tờ cộng lại, trừ phần đè.

In [3]:
# Mô phỏng bằng tập hợp ô đơn vị cho dễ hình dung
A = {(x, y) for x in range(0, 4) for y in range(0, 3)}      # hình 4 x 3 = 12 ô
B = {(x, y) for x in range(2, 6) for y in range(1, 4)}      # hình 4 x 3 = 12 ô
print(len(A), len(B), len(A & B), len(A | B), len(A) + len(B) - len(A & B))   # 12 12 4 20 20

12 12 4 20 20


### 3.3. Phần chồng lấn trên một trục (khoảng giao)

Hai đoạn `[a1, a2]` và `[b1, b2]` trên trục số giao nhau trong đoạn `[max(a1, b1), min(a2, b2)]`, có độ dài `min(a2, b2) - max(a1, b1)`. Nếu giá trị này **âm** thì hai đoạn **không** giao nhau (độ chồng lấn là `0`); nếu bằng `0` thì chúng chỉ **chạm** tại một điểm. Do đó **độ chồng lấn = `max(0, min(a2, b2) - max(a1, b1))`**.

In [4]:
def overlap_1d(a1, a2, b1, b2):
    return max(0, min(a2, b2) - max(a1, b1))


print(overlap_1d(-3, 3, 0, 9))     # 3: giao [0, 3]
print(overlap_1d(0, 2, 2, 5))      # 0: chỉ chạm tại điểm 2
print(overlap_1d(0, 1, 5, 6))      # 0: tách rời (min - max = 1 - 5 = -4 < 0 nên kẹp về 0)
print(overlap_1d(0, 10, 3, 4))     # 1: đoạn nhỏ nằm gọn trong đoạn lớn

3
0
0
1


### 3.4. Phần giao của hai hình chữ nhật

Hai hình giao nhau khi **đồng thời** chồng lấn trên cả hai trục; phần giao là hình chữ nhật có chiều rộng = độ chồng lấn trục `x`, chiều cao = độ chồng lấn trục `y`. Diện tích giao = `overlap_x * overlap_y`. Nếu **một trong hai** bằng `0` thì diện tích giao `= 0`.

In [5]:
def intersection_area(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    return overlap_1d(ax1, ax2, bx1, bx2) * overlap_1d(ay1, ay2, by1, by2)


print(intersection_area(-3, 0, 3, 4, 0, -1, 9, 2))    # 3 * 2 = 6
print(intersection_area(0, 0, 2, 2, 2, 0, 4, 2))      # chỉ chạm cạnh: 0
print(intersection_area(0, 0, 1, 1, 5, 5, 6, 6))      # tách rời: 0

6
0
0


### 3.5. `min`, `max` và gán nhiều biến

`min(a, b)` / `max(a, b)` trả số nhỏ / lớn hơn trong hai số. Python cho phép gán nhiều biến một lúc: `x1, y1, x2, y2 = rect`. Hàm có `8` tham số nên ta thường gom thành hai bộ `(ax1, ay1, ax2, ay2)` và `(bx1, by1, bx2, by2)` khi viết helper, nhưng hàm nộp phải giữ đúng `8` tham số của LeetCode.

In [6]:
rect_a = (-3, 0, 3, 4)
x1, y1, x2, y2 = rect_a
print(x1, y1, x2, y2, max(x1, 0), min(x2, 2))     # -3 0 3 4 0 2

-3 0 3 4 0 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `A = [-3, 3] × [0, 4]`, `B = [0, 9] × [-1, 2]`.

1. Diện tích `A = (3 - (-3)) × (4 - 0) = 6 × 4 = 24`.
2. Diện tích `B = (9 - 0) × (2 - (-1)) = 9 × 3 = 27`.
3. Trục `x`: `A` phủ `[-3, 3]`, `B` phủ `[0, 9]`; giao là `[0, 3]`, dài `3 - 0 = 3`.
4. Trục `y`: `A` phủ `[0, 4]`, `B` phủ `[-1, 2]`; giao là `[0, 2]`, dài `2 - 0 = 2`.
5. Diện tích giao `= 3 × 2 = 6`.
6. Đáp án `= 24 + 27 - 6 = 45`.

Để hình dung, vẽ trục `x` và trục `y` riêng rẽ. **Hai hình chữ nhật cạnh song song trục giao nhau ⟺ hai đoạn chiếu giao nhau trên cả hai trục.** Đây là điều kỳ diệu giúp bài toán hình học 2D thành hai bài toán đoạn thẳng 1D.

## 5. Cách 1 — Brute Force / baseline: đếm các ô đơn vị bằng tập hợp

**Ý tưởng.** Chia mặt phẳng thành các ô đơn vị `1 × 1`. Ô `(x, y)` thuộc hình `[x1, x2) × [y1, y2)` nếu `x1 <= x < x2` và `y1 <= y < y2`. Lập **tập** các ô thuộc `A` và **tập** các ô thuộc `B` rồi lấy **hợp**; số phần tử của hợp là đáp án (phần giao tự động chỉ được đếm một lần vì tập không chứa phần tử trùng).

In [7]:
def area_cells(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    cells = set()
    for x in range(ax1, ax2):
        for y in range(ay1, ay2):
            cells.add((x, y))
    for x in range(bx1, bx2):
        for y in range(by1, by2):
            cells.add((x, y))               # ô đã có thì set bỏ qua: không đếm trùng
    return len(cells)


# Ví dụ 1 có tọa độ nhỏ nên chạy được
for args, expected in examples:
    print(args, "->", area_cells(*args), "(kỳ vọng", expected, ")")

(-3, 0, 3, 4, 0, -1, 9, 2) -> 45 (kỳ vọng 45 )
(-2, -2, 2, 2, -2, -2, 2, 2) -> 16 (kỳ vọng 16 )


**Độ phức tạp (Cách 1).** Gọi `W`, `H` là bề rộng, bề cao của hộp bao chứa cả hai hình. Cách này tạo tới `W·H` ô: **Time `O(W·H)`**, **Space `O(W·H)`**. Với tọa độ tới `10^4` (hộp tối đa `2·10^4 × 2·10^4 = 4·10^8` ô) thì **không khả thi**.

**Nút thắt.** Ta đếm **từng ô** trong khi diện tích có công thức: ta chỉ cần biết **kích thước phần giao**, tính được trong `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bao hàm – loại trừ:** `đáp án = |A| + |B| - |A ∩ B|`.

**Phần giao tính `O(1)`:**

- chiều rộng giao `= min(ax2, bx2) - max(ax1, bx1)`;
- chiều cao giao `= min(ay2, by2) - max(ay1, by1)`;
- nếu **một trong hai** nhỏ hơn hoặc bằng `0` thì **không có phần giao** (diện tích giao `0`).

**Điểm mấu chốt cần nhớ:** phải **kẹp từng chiều** về `>= 0` (`max(0, ...)`) **trước khi nhân**. Nếu hai hình tách rời theo cả hai trục thì cả hai chiều đều âm, và tích của hai số âm là một số **dương** — sẽ trừ nhầm một lượng lớn.

Tổng thể: **một dòng công thức `O(1)`**.

## 7. Các cách tiếp theo

### Cách 2 — Bao hàm – loại trừ với `max(0, …)` (cách nộp chuẩn)

**Giải thích.** Tính `a`, `b` là diện tích mỗi hình; `width`, `height` là độ chồng lấn đã kẹp `>= 0`; trả `a + b - width * height`.
#### Code demo

In [8]:
def area_inclusion_exclusion(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    a = (ax2 - ax1) * (ay2 - ay1)
    b = (bx2 - bx1) * (by2 - by1)
    width = max(0, min(ax2, bx2) - max(ax1, bx1))
    height = max(0, min(ay2, by2) - max(ay1, by1))
    return a + b - width * height


for args, expected in examples:
    print(args, "->", area_inclusion_exclusion(*args), "(kỳ vọng", expected, ")")

(-3, 0, 3, 4, 0, -1, 9, 2) -> 45 (kỳ vọng 45 )
(-2, -2, 2, 2, -2, -2, 2, 2) -> 16 (kỳ vọng 16 )


In [9]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def computeArea(self, ax1: int, ay1: int, ax2: int, ay2: int,
                    bx1: int, by1: int, bx2: int, by2: int) -> int:
        a = (ax2 - ax1) * (ay2 - ay1)
        b = (bx2 - bx1) * (by2 - by1)
        width = max(0, min(ax2, bx2) - max(ax1, bx1))
        height = max(0, min(ay2, by2) - max(ay1, by1))
        return a + b - width * height


print(Solution().computeArea(-3, 0, 3, 4, 0, -1, 9, 2))   # 45

45


**Độ phức tạp Cách 2:** **Time `O(1)`** (một số phép toán cố định), **Space `O(1)`**.

### Cách 3 — Tách trường hợp rõ ràng (có giao / không giao)

**Giải thích.** Trước tiên kiểm tra **hai hình có giao nhau (với diện tích dương) không** bằng 4 điều kiện loại trừ: `B` nằm hoàn toàn bên phải `A` (`bx1 >= ax2`), bên trái (`bx2 <= ax1`), phía trên (`by1 >= ay2`), phía dưới (`by2 <= ay1`). Nếu một điều kiện đúng thì không giao (hoặc chỉ chạm) và đáp án là `a + b`; ngược lại tính phần giao **không cần kẹp** vì chắc chắn dương.
#### Code demo

In [10]:
def area_by_cases(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    a = (ax2 - ax1) * (ay2 - ay1)
    b = (bx2 - bx1) * (by2 - by1)
    if bx1 >= ax2 or bx2 <= ax1 or by1 >= ay2 or by2 <= ay1:
        return a + b                                   # không giao hoặc chỉ chạm
    left, right = max(ax1, bx1), min(ax2, bx2)
    bottom, top = max(ay1, by1), min(ay2, by2)
    return a + b - (right - left) * (top - bottom)     # chắc chắn right > left và top > bottom


for args, expected in examples:
    print(args, "->", area_by_cases(*args), "(kỳ vọng", expected, ")")

(-3, 0, 3, 4, 0, -1, 9, 2) -> 45 (kỳ vọng 45 )
(-2, -2, 2, 2, -2, -2, 2, 2) -> 16 (kỳ vọng 16 )


**Độ phức tạp Cách 3:** **Time `O(1)`**, **Space `O(1)`**. Dài hơn Cách 2 nhưng giúp **thấy rõ** ba tình huống: tách rời, chạm cạnh, chồng lấn.

### Cách 4 — Hàm chồng lấn 1 chiều tái sử dụng

**Giải thích.** Tách ý "chồng lấn trên một trục" thành hàm `overlap_1d` (đã gặp ở phần 3.3) rồi gọi hai lần: một cho trục `x`, một cho trục `y`. Cách này làm code đọc như đề, dễ kiểm tra từng trục riêng, và dễ **mở rộng sang 3D** (thêm trục `z`).
#### Code demo

In [11]:
def area_overlap_helper(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    inter = overlap_1d(ax1, ax2, bx1, bx2) * overlap_1d(ay1, ay2, by1, by2)
    return area_a + area_b - inter


for args, expected in examples:
    print(args, "->", area_overlap_helper(*args), "(kỳ vọng", expected, ")")

(-3, 0, 3, 4, 0, -1, 9, 2) -> 45 (kỳ vọng 45 )
(-2, -2, 2, 2, -2, -2, 2, 2) -> 16 (kỳ vọng 16 )


**Độ phức tạp Cách 4:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 5 — Nén tọa độ (mở rộng được cho `n` hình)

**Giải thích.** Lấy mọi tọa độ `x` (của cả hai hình) đã **sắp xếp và bỏ trùng**: `xs`; tương tự `ys`. Các tọa độ này chia mặt phẳng thành tối đa `3 × 3` **ô lớn**. Với mỗi ô lớn `[xs[i], xs[i+1]] × [ys[j], ys[j+1]]`, nếu nó nằm trong ít nhất một hình thì cộng diện tích của nó. Kỹ thuật này tổng quát cho **`n` hình chữ nhật** (xem Rectangle Area II, LeetCode 850).
#### Code demo

In [12]:
def union_area(rects):
    """Diện tích hợp của nhiều hình chữ nhật (x1, y1, x2, y2) bằng nén tọa độ, O(n^3) cho đơn giản."""
    xs = sorted({r[0] for r in rects} | {r[2] for r in rects})
    ys = sorted({r[1] for r in rects} | {r[3] for r in rects})
    total = 0
    for i in range(len(xs) - 1):
        for j in range(len(ys) - 1):
            x1, x2, y1, y2 = xs[i], xs[i + 1], ys[j], ys[j + 1]
            # ô lớn được phủ nếu nằm trọn trong ít nhất một hình
            if any(r[0] <= x1 and x2 <= r[2] and r[1] <= y1 and y2 <= r[3] for r in rects):
                total += (x2 - x1) * (y2 - y1)
    return total


def area_compress(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    return union_area([(ax1, ay1, ax2, ay2), (bx1, by1, bx2, by2)])


for args, expected in examples:
    print(args, "->", area_compress(*args), "(kỳ vọng", expected, ")")

(-3, 0, 3, 4, 0, -1, 9, 2) -> 45 (kỳ vọng 45 )
(-2, -2, 2, 2, -2, -2, 2, 2) -> 16 (kỳ vọng 16 )


**Độ phức tạp Cách 5:** với `n` hình: **Time `O(n³)`** theo cách viết đơn giản (có `O(n²)` ô lớn, mỗi ô kiểm tra `O(n)` hình); với `n = 2` là hằng số. **Space `O(n)`**.

> Cách 2 là bản nộp chuẩn. Cách 3 giúp bạn hiểu sâu các trường hợp. Cách 5 là bước đệm sang bài `n` hình chữ nhật.

## 8. TRACE từng bước

Ta in từng đại lượng trung gian cho ví dụ 1, ví dụ 2 và hai trường hợp đặc biệt (chạm cạnh, tách rời).

In [13]:
def trace_area(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    a = (ax2 - ax1) * (ay2 - ay1)
    b = (bx2 - bx1) * (by2 - by1)
    raw_w = min(ax2, bx2) - max(ax1, bx1)
    raw_h = min(ay2, by2) - max(ay1, by1)
    width, height = max(0, raw_w), max(0, raw_h)
    inter = width * height
    print(f"A=[{ax1},{ax2}]x[{ay1},{ay2}]  B=[{bx1},{bx2}]x[{by1},{by2}]")
    print(f"  |A| = {a}, |B| = {b}")
    print(f"  trục x: min({ax2},{bx2}) - max({ax1},{bx1}) = {raw_w} -> kẹp {width}")
    print(f"  trục y: min({ay2},{by2}) - max({ay1},{by1}) = {raw_h} -> kẹp {height}")
    print(f"  giao = {width} * {height} = {inter}; đáp án = {a} + {b} - {inter} = {a + b - inter}")
    if raw_w < 0 and raw_h < 0:
        print(f"  (!) cả hai chiều đều âm: nếu quên kẹp, giao sẽ là {raw_w * raw_h} > 0, sai)")
    return a + b - inter


trace_area(-3, 0, 3, 4, 0, -1, 9, 2)
print()
trace_area(-2, -2, 2, 2, -2, -2, 2, 2)
print()
trace_area(0, 0, 2, 2, 2, 0, 4, 2)           # chỉ chạm cạnh
print()
trace_area(0, 0, 1, 1, 5, 5, 6, 6)           # tách rời

A=[-3,3]x[0,4]  B=[0,9]x[-1,2]
  |A| = 24, |B| = 27
  trục x: min(3,9) - max(-3,0) = 3 -> kẹp 3
  trục y: min(4,2) - max(0,-1) = 2 -> kẹp 2
  giao = 3 * 2 = 6; đáp án = 24 + 27 - 6 = 45

A=[-2,2]x[-2,2]  B=[-2,2]x[-2,2]
  |A| = 16, |B| = 16
  trục x: min(2,2) - max(-2,-2) = 4 -> kẹp 4
  trục y: min(2,2) - max(-2,-2) = 4 -> kẹp 4
  giao = 4 * 4 = 16; đáp án = 16 + 16 - 16 = 16

A=[0,2]x[0,2]  B=[2,4]x[0,2]
  |A| = 4, |B| = 4
  trục x: min(2,4) - max(0,2) = 0 -> kẹp 0
  trục y: min(2,2) - max(0,0) = 2 -> kẹp 2
  giao = 0 * 2 = 0; đáp án = 4 + 4 - 0 = 8

A=[0,1]x[0,1]  B=[5,6]x[5,6]
  |A| = 1, |B| = 1
  trục x: min(1,6) - max(0,5) = -4 -> kẹp 0
  trục y: min(1,6) - max(0,5) = -4 -> kẹp 0
  giao = 0 * 0 = 0; đáp án = 1 + 1 - 0 = 2
  (!) cả hai chiều đều âm: nếu quên kẹp, giao sẽ là 16 > 0, sai)


2

Bảng tóm tắt (ví dụ 1):

| đại lượng | công thức | giá trị |
|---|---|---|
| `|A|` | `(3 - (-3)) × (4 - 0)` | 24 |
| `|B|` | `(9 - 0) × (2 - (-1))` | 27 |
| rộng giao | `min(3, 9) - max(-3, 0) = 3 - 0` | 3 |
| cao giao | `min(4, 2) - max(0, -1) = 2 - 0` | 2 |
| giao | `3 × 2` | 6 |
| đáp án | `24 + 27 - 6` | **45** |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (bao hàm – loại trừ có kẹp `max(0, …)`). Đúng `class Solution`, đúng tên method `computeArea` và đủ `8` tham số.

In [14]:
class Solution:
    def computeArea(self, ax1: int, ay1: int, ax2: int, ay2: int,
                    bx1: int, by1: int, bx2: int, by2: int) -> int:
        area_a = (ax2 - ax1) * (ay2 - ay1)
        area_b = (bx2 - bx1) * (by2 - by1)
        width = max(0, min(ax2, bx2) - max(ax1, bx1))       # độ chồng lấn theo x (>= 0)
        height = max(0, min(ay2, by2) - max(ay1, by1))      # độ chồng lấn theo y (>= 0)
        return area_a + area_b - width * height


sol = Solution()
for args, expected in examples:
    assert sol.computeArea(*args) == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Bao hàm – loại trừ:** diện tích phủ của `A ∪ B` bằng `|A| + |B| - |A ∩ B|` vì mỗi điểm trong `A ∩ B` bị cộng hai lần ở `|A| + |B|` và cần đếm một lần.
- **Giao của hai hình chữ nhật cạnh song song trục** là một hình chữ nhật (hoặc rỗng): điểm `(x, y)` thuộc cả hai ⟺ `x ∈ [ax1, ax2] ∩ [bx1, bx2]` **và** `y ∈ [ay1, ay2] ∩ [by1, by2]`. Giao các khoảng là `[max(ax1, bx1), min(ax2, bx2)]` (nếu `max <= min`). Vậy chiều rộng giao là `min - max` khi dương, ngược lại `0`.
- **Vì sao phải kẹp từng chiều:** kẹp `max(0, ·)` đảm bảo từng chiều `>= 0`; khi hai hình tách rời theo **một** trục, chiều đó âm nhưng ta ép về `0` nên tích `= 0` (đúng); khi tách rời theo **cả hai** trục ta cũng ép từng chiều về `0` thay vì nhân hai số âm ra số dương.
- **Chạm cạnh:** chiều rộng giao `= 0` nên tích `= 0` — diện tích phủ bằng tổng hai diện tích (đúng, chỉ chạm không chồng).
- **Hình suy biến** (`x1 = x2` hoặc `y1 = y2`): diện tích `0`, công thức vẫn đúng.
- **Không phụ thuộc thứ tự:** `min`/`max` đối xứng nên không cần giả định `A` nằm bên trái hay bên dưới `B`.
- **Cỡ số:** mỗi diện tích tối đa `(2·10^4)^2 = 4·10^8`, tổng tối đa `8·10^8 < 2^31 - 1`; trong C++/Java vẫn vừa `int`. Python không tràn số.

## 11. Phân tích Time / Space Complexity

- **Cách 1 (ô đơn vị):** Time `O(W·H)` với `W`, `H` là bề rộng, bề cao hộp bao (tối đa `2·10^4` mỗi chiều ⇒ `4·10^8`), Space `O(W·H)` — **quá lớn**.
- **Cách 2, 3, 4:** Time `O(1)`, Space `O(1)`.
- **Cách 5 (nén tọa độ):** với `n = 2` hình: `O(1)`; tổng quát `n` hình `O(n³)` (viết đơn giản) hoặc `O(n² log n)` / `O(n log n)` với quét đường thẳng nâng cao; Space `O(n)`.

Không có "average/worst" khác nhau vì mọi nhánh đều có số phép toán cố định. Đây là bài **toán học/hình học `O(1)`**: điểm khó không nằm ở thuật toán mà ở việc **xét đúng các trường hợp (tách rời, chạm cạnh, chứa nhau)**.

## 12. Bộ test

`oracle` là brute force **độc lập**: lấy **hộp bao** của hai hình rồi **duyệt từng ô đơn vị** trong hộp, đếm ô thuộc `A` hoặc `B` (không dùng tập hợp, không dùng `min/max` của phần giao). Chỉ dùng được khi tọa độ nhỏ. Với tọa độ lớn, ta đối chiếu các cách `O(1)` với nhau và với tính chất toán học.

In [15]:
def oracle(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    """Brute force độc lập: duyệt từng ô đơn vị trong hộp bao, ô (x, y) là [x, x+1) x [y, y+1)."""
    x_lo, x_hi = min(ax1, bx1), max(ax2, bx2)
    y_lo, y_hi = min(ay1, by1), max(ay2, by2)
    count = 0
    for x in range(x_lo, x_hi):
        for y in range(y_lo, y_hi):
            in_a = ax1 <= x < ax2 and ay1 <= y < ay2
            in_b = bx1 <= x < bx2 and by1 <= y < by2
            if in_a or in_b:
                count += 1
    return count


impls = {
    "cells": area_cells,
    "inclusion_exclusion": area_inclusion_exclusion,
    "cases": area_by_cases,
    "helper": area_overlap_helper,
    "compress": area_compress,
    "submit": sol.computeArea,
}

for args, expected in examples:
    assert oracle(*args) == expected
    for key, fn in impls.items():
        assert fn(*args) == expected, (key, args, fn(*args), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên

In [16]:
edge_cases = [
    ((0, 0, 0, 0, 0, 0, 0, 0), 0, "hai điểm trùng: diện tích 0"),
    ((0, 0, 1, 1, 5, 5, 6, 6), 2, "tách rời theo cả hai trục (bẫy hai số âm)"),
    ((0, 0, 2, 2, 2, 0, 4, 2), 8, "chạm nhau ở một cạnh"),
    ((0, 0, 2, 2, 2, 2, 4, 4), 8, "chạm nhau ở một góc"),
    ((0, 0, 4, 4, 1, 1, 3, 3), 16, "B nằm gọn trong A"),
    ((1, 1, 3, 3, 0, 0, 4, 4), 16, "A nằm gọn trong B"),
    ((0, 0, 3, 3, 1, 1, 5, 5), 9 + 16 - 4, "chồng lấn ở góc"),
    ((0, 0, 4, 2, 1, -1, 3, 3), 8 + 8 - 2 * 2, "hình chữ thập: giao 2x2"),
    ((-10000, -10000, 10000, 10000, -10000, -10000, 10000, 10000), 400_000_000, "biên của đề: hai hình cực đại trùng nhau"),
    ((-10000, -10000, 0, 0, 0, 0, 10000, 10000), 200_000_000, "hai hình cực đại chạm góc"),
    ((0, 0, 0, 5, 0, 0, 5, 0), 0, "hai đoạn thẳng suy biến"),
    ((0, 0, 3, 3, 0, 0, 0, 0), 9, "B là một điểm"),
]
for args, expected, label in edge_cases:
    if max(abs(v) for v in args) <= 20:                # oracle chỉ chạy khi tọa độ nhỏ
        assert oracle(*args) == expected, (label, oracle(*args), expected)
    for key, fn in impls.items():
        if key in ("cells",) and max(abs(v) for v in args) > 20:
            continue                                  # cách ô đơn vị không chạy nổi tọa độ lớn
        assert fn(*args) == expected, (key, label, args, fn(*args), expected)
    print("PASS", label)

PASS hai điểm trùng: diện tích 0
PASS tách rời theo cả hai trục (bẫy hai số âm)
PASS chạm nhau ở một cạnh
PASS chạm nhau ở một góc
PASS B nằm gọn trong A
PASS A nằm gọn trong B
PASS chồng lấn ở góc
PASS hình chữ thập: giao 2x2
PASS biên của đề: hai hình cực đại trùng nhau
PASS hai hình cực đại chạm góc
PASS hai đoạn thẳng suy biến
PASS B là một điểm


### 12.2. Tính chất và phản ví dụ

- **Đối xứng:** đổi vai `A` ↔ `B` không đổi đáp án.
- **Tịnh tiến:** dịch cả hai hình cùng một vectơ không đổi đáp án.
- **Kẹp cận:** `max(|A|, |B|) <= đáp án <= |A| + |B|`.
- **Hoán đổi trục:** đổi `x ↔ y` không đổi đáp án.
- **Phản ví dụ** cho lỗi quên kẹp: hai hình **tách rời theo cả hai trục** `(0,0,1,1)` và `(5,5,6,6)` có đáp án `2`, nhưng không kẹp sẽ trừ `(-4) × (-4) = +16`.

In [17]:
def random_rect(lo, hi):
    x1, x2 = sorted((random.randint(lo, hi), random.randint(lo, hi)))
    y1, y2 = sorted((random.randint(lo, hi), random.randint(lo, hi)))
    return (x1, y1, x2, y2)


random.seed(223)
for _ in range(1000):
    a, b = random_rect(-10000, 10000), random_rect(-10000, 10000)
    ans = sol.computeArea(*a, *b)
    area_a, area_b = (a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1])
    assert ans == sol.computeArea(*b, *a)                                       # đối xứng
    dx, dy = random.randint(-100, 100), random.randint(-100, 100)
    sa = (a[0] + dx, a[1] + dy, a[2] + dx, a[3] + dy)
    sb = (b[0] + dx, b[1] + dy, b[2] + dx, b[3] + dy)
    assert ans == sol.computeArea(*sa, *sb)                                     # tịnh tiến
    assert max(area_a, area_b) <= ans <= area_a + area_b                        # kẹp cận
    ta = (a[1], a[0], a[3], a[2])
    tb = (b[1], b[0], b[3], b[2])
    assert ans == sol.computeArea(*ta, *tb)                                     # hoán đổi trục
print("PASS tính chất (đối xứng, tịnh tiến, kẹp cận, hoán đổi trục) trên 1000 cặp ngẫu nhiên")


def sai_quen_kep(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    a = (ax2 - ax1) * (ay2 - ay1)
    b = (bx2 - bx1) * (by2 - by1)
    width = min(ax2, bx2) - max(ax1, bx1)                  # SAI: không kẹp max(0, ...)
    height = min(ay2, by2) - max(ay1, by1)
    return a + b - width * height


bad = (0, 0, 1, 1, 5, 5, 6, 6)
print("phản ví dụ:", bad, "-> đúng", sol.computeArea(*bad), "| quên kẹp:", sai_quen_kep(*bad))
assert sai_quen_kep(*bad) != sol.computeArea(*bad)

PASS tính chất (đối xứng, tịnh tiến, kẹp cận, hoán đổi trục) trên 1000 cặp ngẫu nhiên
phản ví dụ: (0, 0, 1, 1, 5, 5, 6, 6) -> đúng 2 | quên kẹp: -14


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [18]:
random.seed(4223)
# (1) Tọa độ nhỏ: đối chiếu mọi cách (kể cả đếm ô và nén tọa độ) với oracle brute force
for trial in range(2500):
    a, b = random_rect(-6, 6), random_rect(-6, 6)
    truth = oracle(*a, *b)
    for key, fn in impls.items():
        assert fn(*a, *b) == truth, (key, a, b, fn(*a, *b), truth)
print("PASS 2500 cặp hình nhỏ (tọa độ -6..6, gồm cả hình suy biến): mọi cách khớp oracle")

# (2) Tọa độ lớn (cả khoảng của đề): đối chiếu các cách O(1) với nhau
for trial in range(5000):
    a, b = random_rect(-10000, 10000), random_rect(-10000, 10000)
    ref = sol.computeArea(*a, *b)
    for key in ("inclusion_exclusion", "cases", "helper", "compress"):
        assert impls[key](*a, *b) == ref, (key, a, b)
print("PASS 5000 cặp hình lớn: các cách O(1) và nén tọa độ khớp nhau")

PASS 2500 cặp hình nhỏ (tọa độ -6..6, gồm cả hình suy biến): mọi cách khớp oracle


PASS 5000 cặp hình lớn: các cách O(1) và nén tọa độ khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên kẹp `max(0, …)` (hai số âm nhân ra số dương)
Xem `sai_quen_kep` ở phần test: hai hình `(0,0,1,1)` và `(5,5,6,6)` tách rời theo cả hai trục nên `width = -4`, `height = -4`, tích `= +16` bị **trừ** khỏi tổng `2`, cho `-14`. **Code đúng:** `width = max(0, ...)`, `height = max(0, ...)`.

### Lỗi 2: Chỉ kẹp **tích** (`max(0, width * height)`) thay vì từng chiều
#### Code sai

In [19]:
def sai_kep_tich(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    a = (ax2 - ax1) * (ay2 - ay1)
    b = (bx2 - bx1) * (by2 - by1)
    width = min(ax2, bx2) - max(ax1, bx1)
    height = min(ay2, by2) - max(ay1, by1)
    return a + b - max(0, width * height)                    # SAI: kẹp tích, không kẹp từng chiều


print("tách rời hai trục:", sol.computeArea(0, 0, 1, 1, 5, 5, 6, 6), "|", sai_kep_tich(0, 0, 1, 1, 5, 5, 6, 6))
print("tách rời một trục :", sol.computeArea(0, 0, 1, 1, 5, 0, 6, 1), "|", sai_kep_tich(0, 0, 1, 1, 5, 0, 6, 1))

tách rời hai trục: 2 | -14
tách rời một trục : 2 | 2


**Vì sao sai:** nếu chỉ kẹp tích thì khi cả hai chiều âm, tích dương (`16`) vẫn lọt qua `max(0, 16) = 16`. Phải kẹp **từng chiều** trước khi nhân.

### Lỗi 3: Dùng tọa độ thay vì hiệu tọa độ để tính diện tích
#### Code sai

In [20]:
def sai_dien_tich(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    a = ax2 * ay2                                            # SAI: phải là (ax2 - ax1) * (ay2 - ay1)
    b = bx2 * by2
    width = max(0, min(ax2, bx2) - max(ax1, bx1))
    height = max(0, min(ay2, by2) - max(ay1, by1))
    return a + b - width * height


print("đúng:", sol.computeArea(-3, 0, 3, 4, 0, -1, 9, 2), "| sai:", sai_dien_tich(-3, 0, 3, 4, 0, -1, 9, 2))

đúng: 45 | sai: 24


**Vì sao sai:** `ax2 * ay2` là diện tích hình chữ nhật có một góc ở **gốc tọa độ**, không phải hình đã cho (đặc biệt khi `ax1 != 0`). Dùng **hiệu**: `(ax2 - ax1) * (ay2 - ay1)`.

### Lỗi 4: Giả định `A` luôn ở bên trái/dưới `B`
#### Code sai

In [21]:
def sai_gia_dinh_thu_tu(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    a = (ax2 - ax1) * (ay2 - ay1)
    b = (bx2 - bx1) * (by2 - by1)
    width = max(0, ax2 - bx1)                                # SAI: chỉ đúng khi A nằm bên trái B và gối lên B
    height = max(0, ay2 - by1)
    return a + b - width * height


print("A bên trái B  :", sol.computeArea(0, 0, 3, 3, 2, 2, 5, 5), "|", sai_gia_dinh_thu_tu(0, 0, 3, 3, 2, 2, 5, 5))
print("B bên trái A  :", sol.computeArea(2, 2, 5, 5, 0, 0, 3, 3), "|", sai_gia_dinh_thu_tu(2, 2, 5, 5, 0, 0, 3, 3))
print("B nằm trong A :", sol.computeArea(0, 0, 10, 10, 2, 2, 4, 4), "|", sai_gia_dinh_thu_tu(0, 0, 10, 10, 2, 2, 4, 4))

A bên trái B  : 17 | 17
B bên trái A  : 17 | -7
B nằm trong A : 100 | 40


**Vì sao sai:** công thức `ax2 - bx1` chỉ đúng một cấu hình. Dùng **`min` của các cạnh phải trừ `max` của các cạnh trái** để công thức đúng cho mọi vị trí tương đối (đối xứng giữa `A` và `B`).

### Lỗi 5: Chỉ kẹp một chiều (quên kẹp chiều còn lại)
#### Code sai

In [22]:
def sai_kep_mot_chieu(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    a = (ax2 - ax1) * (ay2 - ay1)
    b = (bx2 - bx1) * (by2 - by1)
    width = max(0, min(ax2, bx2) - max(ax1, bx1))
    height = min(ay2, by2) - max(ay1, by1)                   # SAI: quên max(0, ...)
    return a + b - width * height


# A = (0,0)-(4,1), B = (1,3)-(3,4): chồng lấn theo x nhưng tách rời theo y
print("đúng:", sol.computeArea(0, 0, 4, 1, 1, 3, 3, 4), "| sai:", sai_kep_mot_chieu(0, 0, 4, 1, 1, 3, 3, 4))

đúng: 6 | sai: 10


**Vì sao sai:** `width = 2 > 0` nhưng `height = min(1, 4) - max(0, 3) = -2 < 0`: tích `-4` bị trừ khỏi tổng (`6 - (-4) = 10`) — kết quả **lớn hơn** cả tổng hai diện tích, vô lý. Mỗi chiều đều phải được kẹp `>= 0`. Bài học: sau khi viết xong, hãy kiểm tra tính chất `đáp án <= |A| + |B|` (xem phần test).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**diện tích hợp / tổng diện tích phủ**" của các hình → **bao hàm – loại trừ**: tổng các phần trừ giao.
- Thấy "**hai hình chữ nhật cạnh song song trục**" → tách bài toán thành **hai bài toán 1D** (độ chồng lấn của đoạn trên mỗi trục).
- **Độ chồng lấn của hai đoạn** `[a1, a2]`, `[b1, b2]` luôn là `max(0, min(a2, b2) - max(a1, b1))` — thuộc lòng công thức này (dùng lại ở Rectangle Overlap, Merge Intervals, Interval List Intersections...).
- **Kẹp từng chiều** trước khi nhân. Đó là lỗi số một của bài này.
- Muốn mở rộng lên **nhiều hình** → **nén tọa độ** hoặc quét đường thẳng (sweep line).
- Luôn thử 4 ca: **tách rời hai trục**, **chạm cạnh**, **hình này nằm trong hình kia**, **trùng nhau hoàn toàn**.

## 15. Biến thể của bài

1. **Rectangle Overlap** (LeetCode 836): chỉ hỏi hai hình có **chồng diện tích dương** không: `overlap_x > 0 and overlap_y > 0`.
2. **Rectangle Area II** (850): diện tích hợp của `n` hình chữ nhật — nén tọa độ hoặc sweep line.
3. **Perfect Rectangle** (391): các hình có ghép thành hình chữ nhật hoàn hảo không.
4. **Diện tích giao của ba hình**: `min` của ba cạnh phải trừ `max` của ba cạnh trái.
5. **Hình hộp 3D**: thêm trục `z`, thể tích hợp = tổng − giao.
6. **Tìm hình chữ nhật giao** (trả tọa độ thay vì diện tích) hoặc **hộp bao nhỏ nhất** chứa cả hai hình.

In [23]:
def rectangles_overlap(a, b):
    """LeetCode 836: giao có diện tích dương?"""
    return overlap_1d(a[0], a[2], b[0], b[2]) > 0 and overlap_1d(a[1], a[3], b[1], b[3]) > 0


def intersection_rect(a, b):
    """Trả hình chữ nhật giao (x1, y1, x2, y2) hoặc None."""
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    return (x1, y1, x2, y2) if x1 < x2 and y1 < y2 else None


def bounding_box(a, b):
    return (min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3]))


print(rectangles_overlap((0, 0, 2, 2), (1, 1, 3, 3)), rectangles_overlap((0, 0, 1, 1), (1, 0, 2, 1)))   # True False
print(intersection_rect((-3, 0, 3, 4), (0, -1, 9, 2)))                                               # (0, 0, 3, 2)
print(bounding_box((-3, 0, 3, 4), (0, -1, 9, 2)))                                                    # (-3, -1, 9, 4)
print(union_area([(0, 0, 2, 2), (1, 1, 3, 3), (2, 0, 4, 1)]))                                        # 9 = 4 + 4 + 2 - 1 (chỉ A và B chồng nhau 1 ô)

True False
(0, 0, 3, 2)
(-3, -1, 9, 4)
9


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đếm ô đơn vị | lập tập các ô của `A` và `B`, lấy hợp | `O(W·H)` | `O(W·H)` | oracle cho tọa độ nhỏ |
| 2. Bao hàm – loại trừ + `max(0,·)` | `|A| + |B| - |giao|` | `O(1)` | `O(1)` | **cách nộp chuẩn** |
| 3. Tách trường hợp | nếu không giao trả `a + b`, ngược lại trừ giao | `O(1)` | `O(1)` | khi muốn nhìn rõ từng trường hợp |
| 4. Hàm chồng lấn 1D | `overlap_1d` cho từng trục | `O(1)` | `O(1)` | code gọn, mở rộng 3D/nhiều trục |
| 5. Nén tọa độ | chia mặt phẳng bởi các tọa độ, cộng ô được phủ | `O(n³)` (n hình) | `O(n)` | mở rộng ra `n` hình (Rectangle Area II) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `overlap_1d(0, 5, 3, 8)`; (b) `overlap_1d(0, 2, 2, 5)`; (c) `overlap_1d(0, 1, 5, 6)`; (d) `Solution().computeArea(0, 0, 2, 2, 1, 1, 3, 3)`.

**Bài 2 (tính bằng tay).** Tính diện tích phủ của `A = (-1, -1, 2, 3)` và `B = (1, 0, 4, 2)`: ghi `|A|`, `|B|`, độ chồng lấn hai trục, đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `compute_area_todo` theo Cách 2.

In [24]:
def compute_area_todo(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    # TODO 1: area_a = (ax2 - ax1) * (ay2 - ay1), area_b tương tự
    # TODO 2: width = max(0, min(ax2, bx2) - max(ax1, bx1))
    # TODO 3: height = max(0, min(ay2, by2) - max(ay1, by1))
    # TODO 4: trả area_a + area_b - width * height
    pass


# Tự làm xong rồi mới mở test:
# for args, expected in examples:
#     assert compute_area_todo(*args) == expected

**Bài 4 (tự viết lại).** Viết `compute_area_cases2(...)` theo Cách 3 (tách trường hợp), **không nhìn lại** code mẫu; kiểm tra bằng `oracle` trên các hình nhỏ ngẫu nhiên.

**Bài 5 (biến thể / mini challenge).** Viết `area_three(a, b, c)` tính **diện tích hợp của ba hình chữ nhật** bằng **bao hàm – loại trừ**: `|A|+|B|+|C|-|A∩B|-|A∩C|-|B∩C|+|A∩B∩C|`, rồi đối chiếu với `union_area` (nén tọa độ). Gợi ý: giao của ba hình chữ nhật là `intersection_rect` lồng nhau.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: độ chồng lấn là `max(0, min(đầu phải) - max(đầu trái))`. Bài 2: tính từng đại lượng riêng như phần 4. Bài 5: giao ba hình = giao của (giao hai hình) với hình thứ ba.
- **Mức 2 (vừa):** Bài 1: `2`, `0`, `0`, `7` (`4 + 4 - 1`). Bài 2: `|A| = 3 × 4 = 12`, `|B| = 3 × 2 = 6`; trục `x`: `min(2, 4) - max(-1, 1) = 1`; trục `y`: `min(3, 2) - max(-1, 0) = 2`. Bài 3: đúng khung Cách 2.
- **Mức 3 (sâu):** Bài 2: giao `1 × 2 = 2`, đáp án `12 + 6 - 2 = 16`. Bài 4: `if bx1 >= ax2 or bx2 <= ax1 or by1 >= ay2 or by2 <= ay1: return a + b`. Bài 5: `inter = lambda p, q: intersection_rect(p, q)`; nếu `None` thì diện tích `0`; với `A∩B∩C` giao `inter(inter(A, B), C)` (chú ý `None`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
# Bài 1
assert overlap_1d(0, 5, 3, 8) == 2 and overlap_1d(0, 2, 2, 5) == 0 and overlap_1d(0, 1, 5, 6) == 0
assert Solution().computeArea(0, 0, 2, 2, 1, 1, 3, 3) == 7

# Bài 2
assert Solution().computeArea(-1, -1, 2, 3, 1, 0, 4, 2) == 16 == oracle(-1, -1, 2, 3, 1, 0, 4, 2)


# Bài 3
def compute_area_todo_ans(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    width = max(0, min(ax2, bx2) - max(ax1, bx1))
    height = max(0, min(ay2, by2) - max(ay1, by1))
    return area_a + area_b - width * height


for args, expected in examples:
    assert compute_area_todo_ans(*args) == expected


# Bài 4
def compute_area_cases2(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    a = (ax2 - ax1) * (ay2 - ay1)
    b = (bx2 - bx1) * (by2 - by1)
    if bx1 >= ax2 or bx2 <= ax1 or by1 >= ay2 or by2 <= ay1:
        return a + b
    return a + b - (min(ax2, bx2) - max(ax1, bx1)) * (min(ay2, by2) - max(ay1, by1))


random.seed(5)
for _ in range(500):
    a, b = random_rect(-6, 6), random_rect(-6, 6)
    assert compute_area_cases2(*a, *b) == oracle(*a, *b)


# Bài 5
def rect_area(r):
    return 0 if r is None else (r[2] - r[0]) * (r[3] - r[1])


def area_three(a, b, c):
    ab = intersection_rect(a, b)
    ac = intersection_rect(a, c)
    bc = intersection_rect(b, c)
    abc = intersection_rect(ab, c) if ab is not None else None
    return (rect_area(a) + rect_area(b) + rect_area(c)
            - rect_area(ab) - rect_area(ac) - rect_area(bc) + rect_area(abc))


random.seed(6)
for _ in range(500):
    a, b, c = random_rect(-8, 8), random_rect(-8, 8), random_rect(-8, 8)
    assert area_three(a, b, c) == union_area([a, b, c])
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (bao hàm – loại trừ, kẹp từng chiều)

```python
class Solution:
    def computeArea(self, ax1, ay1, ax2, ay2, bx1, by1, bx2, by2) -> int:
        a = (ax2 - ax1) * (ay2 - ay1)
        b = (bx2 - bx1) * (by2 - by1)
        width = max(0, min(ax2, bx2) - max(ax1, bx1))
        height = max(0, min(ay2, by2) - max(ay1, by1))
        return a + b - width * height
```

**Checklist**

- [ ] Diện tích dùng **hiệu** tọa độ `(x2 - x1) * (y2 - y1)`?
- [ ] Độ chồng lấn mỗi trục là `max(0, min(đầu phải) - max(đầu trái))`, **kẹp từng chiều trước khi nhân**?
- [ ] Đáp án `= |A| + |B| - giao` (bao hàm – loại trừ)?
- [ ] Công thức đối xứng `A ↔ B` (không giả định hình nào ở trái/dưới)?
- [ ] Đã thử: tách rời hai trục, chạm cạnh, chạm góc, lồng nhau, trùng nhau, hình suy biến?
- [ ] Đã đối chiếu với brute force đếm ô đơn vị trên hình nhỏ ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def computeArea(self, ax1, ay1, ax2, ay2, bx1, by1, bx2, by2) -> int` (8 tham số)?